# 00 — Data Inventory

**What this notebook does:** establishes, from the files themselves, exactly what
data this project holds. Nothing here is copied from an earlier document; every
number is measured.

**Why it runs first:** the previous two versions of this project made decisions
based on inventory figures that turned out to be wrong — a 1 m elevation model
was listed as "missing" when it was sitting in `data/`, and the class-imbalance
figure was out by 25%. Measuring first is cheap. Being wrong for six months is
not.

**Runtime:** about 2 minutes. It reads all 28 raw CSVs (54.8 GB) but only counts
rows and distinct stations, so it never loads them into memory.

**Outputs, written to `docs/reports/phase0/`:**

| File | Contents |
|---|---|
| `inventory_station_years.csv` | one row per sensor per year: counts and time range |
| `inventory_files.csv` | every raw file: size, rows, stations, duplicates |
| `inventory_stations_by_year.csv` | station counts per dataset per year |
| `inventory_gis.csv` | the spatial assets, with real CRS and extents |
| `inventory_registry.csv` | station registry by sensor type and coordinate quality |
| `inventory_prefix_coverage.csv` | which datasets can reach which flood districts |
| `inventory_gaps.csv` | what we do not have, and what each gap costs |

## Setup

In [1]:
import os, sys, json
from pathlib import Path

# Make the shared library importable whether you launched Jupyter from the repo
# root or from notebooks/.
_here = Path.cwd()
_root = next(p for p in [_here, *_here.parents] if (p / "config/config.yaml").is_file())
sys.path.insert(0, str(_root / "src"))
os.chdir(_root)

import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

import bkkflood as bf
from bkkflood.config import load_config
from bkkflood.rawio import DATASETS, connect, raw_file, read_raw_sql

CFG = load_config()
REPORTS = Path(CFG["paths"]["reports"]) / "phase0"
REPORTS.mkdir(parents=True, exist_ok=True)

print(f"repo root      : {_root}")
print(f"config version : {CFG['version']}")
print(f"years          : {CFG['data']['years']}")
print(f"tiers (cm)     : {CFG['flood_event']['tiers_cm']}")
print(f"reports go to  : {REPORTS}")

repo root      : /sessions/compassionate-busy-dijkstra/mnt/bkk-flood-forecast
config version : 3.0.0
years          : [2019, 2020, 2021, 2022, 2023, 2024, 2025]
tiers (cm)     : {'nuisance': 5, 'advisory': 15, 'severe': 30}
reports go to  : docs/reports/phase0


## 1. The raw files

Four datasets, seven years, one CSV per year. The filenames are not consistent —
rain switches from `2019.csv` to `Rain 2021.csv` part-way through — which is why
`config.yaml` carries a per-dataset filename pattern instead of assuming one.

In [2]:
rows = []
for ds in DATASETS:
    for year in CFG["data"]["years"]:
        p = raw_file(ds, year)
        n = p.stat().st_size
        rows.append({"dataset": ds, "year": year, "filename": p.name,
                     "bytes": n,
                     "size_gib": round(n / 1024**3, 3),   # what `du` and Finder show
                     "size_gb": round(n / 1e9, 3)})       # what `ls -l` arithmetic gives
files = pd.DataFrame(rows)
print(f"{len(files)} raw files")
print(f"  {files.size_gib.sum():.1f} GiB  (binary, what Finder and du report)")
print(f"  {files.size_gb.sum():.1f} GB   (decimal, 10^9 bytes)")
print()
files.pivot(index="year", columns="dataset", values="size_gib")

28 raw files
  51.1 GiB  (binary, what Finder and du report)
  54.9 GB   (decimal, 10^9 bytes)



dataset,flood,flow,rain,water
year,,,,
2019,1.096,0.402,1.803,3.686
2020,1.099,0.404,1.809,3.696
2021,1.121,0.402,1.805,3.687
2022,1.138,0.402,1.823,3.789
2023,1.203,0.402,1.822,3.841
2024,1.206,0.403,1.827,4.392
2025,1.203,0.401,1.824,4.423


### The three quirks that will corrupt your data if you ignore them

1. **A UTF-8 byte-order mark on every header line.** Read it naively and the
   first column comes back named `\ufeffrain_code`, so every lookup by name
   fails.
2. **Missing values are the literal text `NULL`.** A column that is mostly
   missing gets parsed as text instead of numbers, and silently stops being
   usable.
3. **The `*_name` columns are Thai free text containing commas.** Split on
   commas and every field after the name shifts by one.

`bkkflood.rawio` handles all three. The cell below shows the raw bytes so the
problem is visible rather than merely described.

In [3]:
p = raw_file("rain", 2019)
with open(p, "rb") as fh:
    head = fh.read(220)
print("raw bytes  :", head[:45])
print()
print("decoded    :")
print(head.decode("utf-8-sig")[:170])
print()
# A water row, where the Thai station name contains a comma:
with open(raw_file("water", 2019), encoding="utf-8-sig") as fh:
    fh.readline()
    print("a water row:", fh.readline().strip())

raw bytes  : b'\xef\xbb\xbfrain_code,rain_name,site_timestamp,rf5min,'

decoded    :
rain_code,rain_name,site_timestamp,rf5min,rf15min,rf30min,rf1hr,rf3hr,rf6hr,rf12hr,rf24hr
RF.BBN.01,สำนักงานเขตบางบอน,2019-01-01 00:00:00.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0

a water row: WL.BBN.01,จุดวัดคลองบางบอน ตอนถนนบางขุนเทียน,2019-01-01 00:00:00.000,-0.01,NULL,NULL


## 2. What is actually in them

One streaming pass per file, grouped by station. From that single pass we get row
counts, station counts, the time range and the duplicate check — no file is ever
loaded into memory, and nothing is read twice.

**The check that matters most is `duplicate_timestamps`.** If a station reports
the same timestamp twice, every lag feature and every event boundary computed
later is wrong, and the failure is silent.

In [4]:
con = connect()
profiles = []
for ds in DATASETS:
    code = CFG["data"]["schema"][ds]["code"]
    for year in CFG["data"]["years"]:
        src = read_raw_sql(ds, year)
        q = (
            "SELECT " + code + " AS station_code, "
            "count(*)::BIGINT AS rows, "
            "count(DISTINCT site_timestamp)::BIGINT AS n_timestamps, "
            "min(site_timestamp) AS ts_min, max(site_timestamp) AS ts_max "
            "FROM " + src + " GROUP BY 1"
        )
        df = con.execute(q).fetchdf()
        df.insert(0, "year", year)
        df.insert(0, "dataset", ds)
        profiles.append(df)
        print(f"  {ds:<6} {year}  rows={int(df['rows'].sum()):>12,}  "
              f"stations={len(df):>4}", flush=True)

prof = pd.concat(profiles, ignore_index=True)
prof["duplicate_timestamps"] = prof["rows"] - prof["n_timestamps"]
prof.to_csv(REPORTS / "inventory_station_years.csv", index=False)
print()
print(f"{len(prof):,} station-years profiled")

  flood  2019  rows=  10,406,880  stations=  99


  flood  2020  rows=  10,435,392  stations=  99


  flood  2021  rows=  10,593,504  stations= 102


  flood  2022  rows=  10,722,240  stations= 102


  flood  2023  rows=  11,247,840  stations= 107


...

  water  2022  rows=  27,541,440  stations= 262


  water  2023  rows=  27,856,800  stations= 265


  water  2024  rows=  31,306,176  stations= 297


  water  2025  rows=  31,536,000  stations= 300



3,736 station-years profiled


In [5]:
inv = (prof.groupby(["dataset", "year"])
            .agg(rows=("rows", "sum"),
                 stations=("station_code", "nunique"),
                 duplicate_timestamps=("duplicate_timestamps", "sum"),
                 ts_min=("ts_min", "min"),
                 ts_max=("ts_max", "max"))
            .reset_index()
            .merge(files[["dataset", "year", "filename", "size_gib", "size_gb"]],
                   on=["dataset", "year"]))
inv.to_csv(REPORTS / "inventory_files.csv", index=False)

total_dupes = int(inv.duplicate_timestamps.sum())
print(f"TOTAL ROWS              : {inv['rows'].sum():,}")
print(f"DUPLICATE (station, ts) : {total_dupes:,}")
assert total_dupes == 0, "Duplicate timestamps found - every lag feature downstream is unsafe."
print("OK: the time grid is clean.")

TOTAL ROWS              : 392,909,188
DUPLICATE (station, ts) : 0
OK: the time grid is clean.


### Is the grid complete?

A healthy sensor produces one reading every 5 minutes, so 365 x 24 x 12 = 105,120
rows a year (105,408 in a leap year). Comparing that against what is actually
present separates two very different problems: *missing rows* (the export dropped
data) from *missing values* (the sensor was offline but the row still exists).

In [6]:
def expected_rows(year, cadence=CFG["data"]["cadence_minutes"]):
    days = 366 if (year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)) else 365
    return days * 24 * 60 // cadence

prof["expected_rows"] = prof["year"].map(expected_rows)
prof["row_completeness_pct"] = (100 * prof["rows"] / prof["expected_rows"]).round(2)

incomplete = prof[prof.row_completeness_pct < 100].sort_values("row_completeness_pct")
print(f"station-years that are NOT exactly complete: {len(incomplete)} of {len(prof)}")
print(f"missing rows in total: {int((incomplete.expected_rows - incomplete['rows']).sum()):,} "
      f"({100 * (incomplete.expected_rows - incomplete['rows']).sum() / prof['rows'].sum():.4f}% of the archive)")
print()
display(incomplete[["dataset", "year", "station_code", "rows", "expected_rows",
                    "row_completeness_pct", "ts_min", "ts_max"]])

station-years that are NOT exactly complete: 6 of 3736
missing rows in total: 129,884 (0.0331% of the archive)



,dataset,year,station_code,rows,expected_rows,row_completeness_pct,ts_min,ts_max
203,flood,2021,FL.MBR.01,40320,105120,38.36,2021-08-14,2021-12-31 23:55:00
200,flood,2021,FL.DST.08,41184,105120,39.18,2021-08-11,2021-12-31 23:55:00
1375,rain,2022,RF.PYT.02,104833,105120,99.73,2022-01-01,2022-12-31 00:00:00
1537,rain,2023,RF.PYT.02,104833,105120,99.73,2023-01-01,2023-12-31 00:00:00
1604,rain,2024,RF.PYT.02,105121,105408,99.73,2024-01-01,2024-12-31 00:00:00
1778,rain,2025,RF.PYT.02,104833,105120,99.73,2025-01-01,2025-12-31 00:00:00


Six exceptions in 3,736 station-years, and they are two different stories — worth
separating, because one is a non-issue and the other is a small data defect.

**FL.MBR.01 and FL.DST.08 in 2021 (38–39% complete).** Look at `ts_min`: these
sensors start reporting on 11 and 14 August 2021. They were *installed* mid-year.
Nothing is missing; the sensor did not exist yet. Any per-station statistic for
2021 must be computed over the period the station was alive, not over the calendar
year, or these two will look like catastrophic outages.

**RF.PYT.02 in 2022, 2023, 2024 and 2025 (99.73% complete).** Exactly 287 rows
short every year, and `ts_max` lands on 31 December at `00:00:00` instead of
`23:55:00`. The final day is truncated at midnight. 287 = 288 − 1, i.e. one whole
day minus the single row that survived. This is an export bug on one rain gauge,
it repeats every year, and it is worth mentioning to BMA — but at 0.00003% of the
archive it changes nothing.

> **Correction to spec §B.3.** That section states every station-year has exactly
> 105,120 rows. That is true for 3,730 of 3,736 station-years, not all of them.
> The headline conclusion still holds — the time grid is essentially complete and
> has no duplicates — but "exactly" was too strong, and it is fixed here rather
> than left for someone to trip over.

Missing data in this archive is therefore almost entirely missing **values inside
present rows**, not missing rows. Notebook 02 measures that, and it turns out to
get substantially worse after 2022.

In [7]:
print("Rows per year (millions)")
display(inv.pivot(index="year", columns="dataset", values="rows").div(1e6).round(2))

print()
print("Stations per year")
by_year = inv.pivot(index="year", columns="dataset", values="stations")
by_year.to_csv(REPORTS / "inventory_stations_by_year.csv")
display(by_year)

print()
print("Total rows per dataset")
display(inv.groupby("dataset")["rows"].sum().apply(lambda v: f"{v:,}"))

Rows per year (millions)


dataset,flood,flow,rain,water
year,,,,
2019,10.41,3.15,13.67,26.81
2020,10.44,3.16,13.70,26.88
2021,10.59,3.15,13.67,26.81
2022,10.72,3.15,13.77,27.54
2023,11.25,3.15,13.77,27.86
2024,11.28,3.16,13.81,31.31
2025,11.25,3.15,13.77,31.54



Stations per year


dataset,flood,flow,rain,water
year,,,,
2019,99,30,130,255
2020,99,30,130,255
2021,102,30,130,255
2022,102,30,131,262
2023,107,30,131,265
2024,107,30,131,297
2025,107,30,131,300



Total rows per dataset


dataset
flood     75,932,352
flow      22,092,480
rain      96,153,700
water    198,730,656
Name: rows, dtype: object

**The sensor network grows.** Flood sensors go 99 to 107, and water 255 to 300,
over the seven years. That is good news operationally and a problem for
modelling: a model that leans on station identity has no history for a sensor
installed last year. This is the cold-start problem, and it is why terrain
features (Phase 1) matter more than they first appear.

In [8]:
n_years = len(CFG["data"]["years"])
years_seen = prof.groupby(["dataset", "station_code"])["year"].nunique()
part = []
for ds in DATASETS:
    s = years_seen.loc[ds]
    partial = s[s < n_years]
    part.append({"dataset": ds, "stations_total": len(s),
                 "in_all_years": int((s == n_years).sum()),
                 "partial_history": len(partial),
                 "examples": ", ".join(sorted(partial.index)[:4])})
pd.DataFrame(part)

,dataset,stations_total,in_all_years,partial_history,examples
0,flood,107,99,8,"FL.BKA.02, FL.BKT.01, FL.BNA.05, FL.DST.08"
1,flow,30,30,0,
2,rain,131,130,1,RF.PYT.02
3,water,300,255,45,"WL.AJP.01, WL.ANX.01, WL.BAM.01, WL.BKT.01"


## 3. Station codes, and the spatial join

Every code looks like `TYPE.PREFIX.NN`, e.g. `FL.BBN.01`.

For **rain** and **flood** the middle part is a *district* abbreviation
(BBN = Bang Bon). For **water** and **flow** it is a *canal* name abbreviation
(SSB = Saen Saep). Those are two different naming systems, and the consequence is
severe: rainfall can be joined to a flood site by district, but canal level and
flow cannot, so both can only enter the model as citywide averages.

In [9]:
from bkkflood.stations import prefix_coverage, prefixes

codes = {ds: sorted(prof.loc[prof.dataset == ds, "station_code"].unique())
         for ds in DATASETS}
cov = []
for ds in ["rain", "water", "flow"]:
    c = prefix_coverage(codes["flood"], codes[ds])
    cov.append({"dataset": ds,
                "own_prefixes": len(prefixes(codes[ds])),
                "covers_flood_districts": c["covered"],
                "of_total": c["flood_prefixes"],
                "pct": c["pct"],
                "cannot_reach": ", ".join(c["uncovered"]) or "(none)"})
coverage = pd.DataFrame(cov)
coverage.to_csv(REPORTS / "inventory_prefix_coverage.csv", index=False)
print(f"Flood sensors span {len(prefixes(codes['flood']))} district prefixes.")
print()
coverage

Flood sensors span 33 district prefixes.



,dataset,own_prefixes,covers_flood_districts,of_total,pct,cannot_reach
0,rain,51,33,33,100.0,(none)
1,water,163,13,33,39.4,"BKE, BKM, BKN, BKP, BRK, DDG, DST, JTG, KSN, L..."
2,flow,27,3,33,9.1,"BBN, BKE, BKL, BKM, BKN, BKP, BKT, BNA, BRK, C..."


> **Read that table again.** Rainfall reaches 100% of the districts that have
> flood sensors. Canal water level reaches 39%, and flow reaches 9%. Two thirds of
> the canal network cannot be tied to the roads it drains — not because the data
> is bad, but because we do not have a coordinate for each sensor.
>
> This is the second-highest-value item on the BMA data request (spec §D.6), and
> the fix is a spreadsheet, not a model.

## 4. The station registry — and why it must be used carefully

In [10]:
from bkkflood.stations import load_registry, registry_summary

reg = load_registry()
summary = registry_summary(reg)
summary.to_csv(REPORTS / "inventory_registry.csv", index=False)

print(f"{len(reg)} sensors in the registry; "
      f"{int(reg.has_coords.sum())} have any coordinate at all.")
print()
display(summary.pivot(index="sensor_type", columns="coord_quality",
                      values="stations").fillna(0).astype(int))

568 sensors in the registry; 401 have any coordinate at all.



coord_quality,district_centroid,inferred_other,none,subdistrict_centroid
sensor_type,,,,
flood,107,0,0,0
flow,17,2,9,2
rain,126,3,2,0
water,58,31,156,55


In [11]:
# Proof of the caveat: every flood sensor in one district shares one coordinate.
bbn = reg[(reg.sensor_type == "flood") & (reg.station_code.str.contains(".BBN.", regex=False))]
display(bbn[["station_code", "district", "lat", "lon", "coord_quality"]])
print("distinct coordinates among these sensors:",
      bbn[["lat", "lon"]].drop_duplicates().shape[0])

,station_code,district,lat,lon,coord_quality
0,FL.BBN.01,Bang Bon,13.646,100.37,district_centroid
1,FL.BBN.02,Bang Bon,13.646,100.37,district_centroid


distinct coordinates among these sensors: 1


> These are **not surveyed positions**. They were inferred from code prefixes in
> an earlier version of the project: "district centroid" means every sensor in
> the district carries the same point, and on a map they stack on top of each
> other.
>
> Fine for a district choropleth. Not fine for distance features, spatial
> interpolation, or the "continuous flood surface" the dashboard scheme asks for —
> see spec §C.2 for why that panel is rejected on evidence.
>
> Whenever these coordinates reach the API they must travel with `coord_quality`
> attached, so the frontend can draw a dashed marker instead of implying a
> precision we do not have.

## 5. Spatial assets — measured, not assumed

In [12]:
import rasterio
from rasterio.warp import transform_bounds

gis_rows = []
for label, key in [("1 m DTM", "dtm_1m"), ("SRTM DEM", "dem_srtm")]:
    path = Path(CFG["paths"][key])
    with rasterio.open(path) as s:
        b = transform_bounds(s.crs, "EPSG:4326", *s.bounds)
        gis_rows.append({
            "asset": label, "path": str(path),
            "size_gb": round(path.stat().st_size / 1024**3, 2),
            "crs": str(s.crs), "resolution_m": round(float(s.res[0]), 4),
            "width": s.width, "height": s.height,
            "overviews": len(s.overviews(1)),
            "lon_min": round(b[0], 4), "lat_min": round(b[1], 4),
            "lon_max": round(b[2], 4), "lat_max": round(b[3], 4),
        })

def geo_bounds(geojson_path):
    g = json.load(open(geojson_path))
    xs, ys = [], []
    def walk(c):
        if isinstance(c[0], (int, float)):
            xs.append(c[0]); ys.append(c[1])
        else:
            for i in c:
                walk(i)
    for f in g["features"]:
        walk(f["geometry"]["coordinates"])
    return len(g["features"]), min(xs), min(ys), max(xs), max(ys)

n_d, *bkk = geo_bounds(CFG["paths"]["districts"])
n_s, *_ = geo_bounds(CFG["paths"]["subdistricts"])

gis = pd.DataFrame(gis_rows)
gis.to_csv(REPORTS / "inventory_gis.csv", index=False)
display(gis[["asset", "size_gb", "crs", "resolution_m", "overviews",
             "lon_min", "lat_min", "lon_max", "lat_max"]])
print()
print(f"Bangkok boundary: {n_d} districts, {n_s} sub-districts")
print(f"  extent: {bkk[0]:.4f}, {bkk[1]:.4f}  ->  {bkk[2]:.4f}, {bkk[3]:.4f}")

,asset,size_gb,crs,resolution_m,overviews,lon_min,lat_min,lon_max,lat_max
0,1 m DTM,12.94,EPSG:32647,1.0000,8,100.3252,13.4809,100.9407,13.9578
1,SRTM DEM,0.00,EPSG:4326,0.0003,0,100.2215,13.4504,100.9954,14.0085



Bangkok boundary: 50 districts, 169 sub-districts
  extent: 100.3279, 13.4934  ->  100.9385, 13.9546


In [13]:
dtm = gis[gis.asset == "1 m DTM"].iloc[0]
covers = (dtm.lon_min <= bkk[0] and dtm.lat_min <= bkk[1]
          and dtm.lon_max >= bkk[2] and dtm.lat_max >= bkk[3])
print("Does the 1 m DTM cover the whole Bangkok boundary? ->", covers)
assert covers, "DTM does not cover Bangkok - Phase 1 terrain work would be partial."

Does the 1 m DTM cover the whole Bangkok boundary? -> True


> **This overturns a claim made in every earlier document in this project.**
> `docs/technical_roadmap.md` and `docs/data_requests.md` both list
> high-resolution road elevation as *missing data to request from BMA*, and
> describe the terrain features as limited by a 31 m SRTM model that "cannot
> resolve road dips of 20–50 cm".
>
> We already hold a 13.9 GB, **1 metre** DTM covering the entire city. It was
> never used, most likely because it is large and in UTM 47N while everything else
> is in WGS84. That is a processing job (Phase 1), not a data request.
>
> It also supplies the "Road Elevation" half of the supervisor's formula
> `Flood Depth = Water Level − Road Elevation`. The half still missing is the
> sensor coordinates and the water-level datum.

## 6. What we do not have, and what each gap costs

In [14]:
gaps = pd.DataFrame([
 ("Radar rainfall (TMD)", 1,
  "Our rain is a district average; Bangkok floods from 2-5 km cells. Rainfall carries ~76% of the forecasting signal.",
  "Registration with TMD"),
 ("Station coordinates + datum", 2,
  "Turns canal level and flow from citywide averages into local features; makes the map real; makes the depth formula computable.",
  "Likely an existing asset register"),
 ("Canal network topology", 3,
  "Which canal drains into which. Unlocks upstream features and, later, graph models.",
  "BMA Drainage GIS export"),
 ("Pump and gate operation logs", 4,
  "The drainage system has an operator. A model blind to pump state is predicting a system while ignoring its controller.",
  "SCADA extraction"),
 ("Measured Chao Phraya tide", 5,
  "High tide holds the drainage gates shut. We can reconstruct tidal phase from lunar periods but not height.",
  "Hydrographic Dept / RID"),
 ("Independent flood reports", 6,
  "Labels are sensor-only: a flood where there is no sensor never happened, as far as model and evaluation are concerned.",
  "Traffy Fondue archive (NECTEC)"),
 ("2026 sensor data", 7,
  "The 2025 holdout has already been used. A fresh year restores a genuinely sealed test.",
  "File transfer"),
], columns=["missing", "priority", "why_it_matters", "how_to_get_it"])
gaps.to_csv(REPORTS / "inventory_gaps.csv", index=False)
gaps[["priority", "missing", "how_to_get_it"]]

,priority,missing,how_to_get_it
0,1,Radar rainfall (TMD),Registration with TMD
1,2,Station coordinates + datum,Likely an existing asset register
2,3,Canal network topology,BMA Drainage GIS export
3,4,Pump and gate operation logs,SCADA extraction
4,5,Measured Chao Phraya tide,Hydrographic Dept / RID
5,6,Independent flood reports,Traffy Fondue archive (NECTEC)
6,7,2026 sensor data,File transfer


## 7. Summary

In [15]:
print("=" * 74)
print("DATA INVENTORY - VERIFIED")
print("=" * 74)
last_year = max(CFG["data"]["years"])
for ds in DATASETS:
    sub = inv[inv.dataset == ds]
    last = sub[sub.year == last_year].iloc[0]
    print(f"  {ds:<6} {sub['rows'].sum():>13,} rows | "
          f"{int(last['stations']):>3} stations in {last_year} | "
          f"{sub['size_gib'].sum():>5.1f} GiB")
print("-" * 74)
print(f"  {'TOTAL':<6} {inv['rows'].sum():>13,} rows | "
      f"{len(reg):>3} sensors in registry | {inv['size_gib'].sum():>5.1f} GiB")
print("=" * 74)
print(f"  duplicate (station, timestamp) pairs   : {total_dupes}")
print(f"  station-years not exactly complete     : {len(incomplete)} of {len(prof)}")
print(f"  sensors with any coordinate            : {int(reg.has_coords.sum())} / {len(reg)}")
print(f"  sensors with a SURVEYED coordinate     : 0 / {len(reg)}   <- all inferred")
print(f"  1 m DTM covers Bangkok                 : {covers}")
print("=" * 74)
print()
print("Next: notebook 01 turns these CSVs into clean Parquet.")
con.close()

DATA INVENTORY - VERIFIED
  flood     75,932,352 rows | 107 stations in 2025 |   8.1 GiB
  flow      22,092,480 rows |  30 stations in 2025 |   2.8 GiB
  rain      96,153,700 rows | 131 stations in 2025 |  12.7 GiB
  water    198,730,656 rows | 300 stations in 2025 |  27.5 GiB
--------------------------------------------------------------------------
  TOTAL    392,909,188 rows | 568 sensors in registry |  51.1 GiB
  duplicate (station, timestamp) pairs   : 0
  station-years not exactly complete     : 6 of 3736
  sensors with any coordinate            : 401 / 568
  sensors with a SURVEYED coordinate     : 0 / 568   <- all inferred
  1 m DTM covers Bangkok                 : True

Next: notebook 01 turns these CSVs into clean Parquet.
